In [1]:
import subprocess
subprocess.run(["pip", "install", "einops", "nibabel", "-q"])

import os, json, time, gc, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from scipy.ndimage import zoom, rotate, gaussian_filter, map_coordinates
from scipy.ndimage import label as scipy_label
import nibabel as nib
from einops import rearrange
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

Device: cuda
GPU: Tesla P100-PCIE-16GB


In [2]:
DATA_DIR = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas"
IMAGES_DIR = os.path.join(DATA_DIR, "imagesTr")
LABELS_DIR = os.path.join(DATA_DIR, "labelsTr")

# Find checkpoint
CHECKPOINT_PATH = None
for root in ["/kaggle/input", "/kaggle/working"]:
    if not os.path.exists(root): continue
    for dp, dn, fn in os.walk(root):
        for f in fn:
            if f.endswith('.pth') and 'stage1' in f:
                candidate = os.path.join(dp, f)
                print(f"  Found: {candidate}")
                if 'ep99' in f:
                    CHECKPOINT_PATH = candidate

# Verify
imgs = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith(('.nii', '.nii.gz'))])
lbls = sorted([f for f in os.listdir(LABELS_DIR) if f.endswith(('.nii', '.nii.gz'))])
print(f"\n✅ Images: {len(imgs)} in {IMAGES_DIR}")
print(f"✅ Labels: {len(lbls)} in {LABELS_DIR}")
print(f"✅ Checkpoint: {CHECKPOINT_PATH}")

  Found: /kaggle/input/models/logiverse/model1ep99/pytorch/default/1/stage1_ep99.pth
  Found: /kaggle/input/models/logiverse/model1best/pytorch/default/1/stage1_best.pth

✅ Images: 283 in /kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas/imagesTr
✅ Labels: 282 in /kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas/labelsTr
✅ Checkpoint: /kaggle/input/models/logiverse/model1ep99/pytorch/default/1/stage1_ep99.pth


In [3]:
CONFIG = {
    "patch_size": (96, 96, 96),
    "batch_size": 4,
    "accum_steps": 2,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "epochs": 250,
    "val_split": 0.15,
    "num_workers": 2,
    "fg_sample_rate": 0.7,
    "target_spacing": (1.5, 1.5, 2.5),
    "hu_window": (-125, 275),
}

CACHE_DIR = "/kaggle/working/cache_data"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(CACHE_DIR, exist_ok=True)
print("✅ Config ready")
print(f"   Resume from: {CHECKPOINT_PATH}")
print(f"   Train epochs 100 → {CONFIG['epochs']}")

✅ Config ready
   Resume from: /kaggle/input/models/logiverse/model1ep99/pytorch/default/1/stage1_ep99.pth
   Train epochs 100 → 250


In [4]:
# Get all image files
# Filter out Mac junk files (._xxx)
img_files = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith(('.nii', '.nii.gz')) and not f.startswith('._')])
lbl_files = sorted([f for f in os.listdir(LABELS_DIR) if f.endswith(('.nii', '.nii.gz')) and not f.startswith('._')])
# Match images to labels
cases = []
for img_f in img_files:
    name = img_f.replace(".nii.gz", "").replace(".nii", "")
    # Try to find matching label
    for lbl_f in lbl_files:
        if name in lbl_f:
            cases.append((
                os.path.join(IMAGES_DIR, img_f),
                os.path.join(LABELS_DIR, lbl_f)
            ))
            break

print(f"Found {len(cases)} matched cases. Preprocessing...")

cached_files = []
for i, (img_path, lbl_path) in enumerate(cases):
    name = os.path.basename(img_path).replace(".nii.gz", "").replace(".nii", "")
    cache_path = os.path.join(CACHE_DIR, f"{name}.npz")

    if os.path.exists(cache_path):
        cached_files.append(cache_path)
        continue

    ct = nib.load(img_path).get_fdata().astype(np.float32)
    label = nib.load(lbl_path).get_fdata().astype(np.int32)
    spacing = np.array(nib.load(img_path).header.get_zooms()[:3])

    scale = spacing / np.array(CONFIG["target_spacing"])
    ct = zoom(ct, scale, order=1)
    label = zoom(label, scale, order=0)
    ct = np.clip(ct, CONFIG["hu_window"][0], CONFIG["hu_window"][1])

    fg = label > 0
    if fg.sum() > 100:
        ct = (ct - ct[fg].mean()) / (ct[fg].std() + 1e-8)
    else:
        ct = (ct - ct.mean()) / (ct.std() + 1e-8)

    np.savez_compressed(cache_path,
        image=ct.astype(np.float32),
        label=label.astype(np.int8),
        shape=np.array(ct.shape)
    )
    cached_files.append(cache_path)

    if (i+1) % 30 == 0:
        print(f"  [{i+1}/{len(cases)}] {name} shape={ct.shape}")

print(f"\n✅ {len(cached_files)} cached files ready")

Found 281 matched cases. Preprocessing...
  [30/281] pancreas_055 shape=(280, 280, 96)
  [60/281] pancreas_100 shape=(280, 280, 98)
  [90/281] pancreas_148 shape=(260, 260, 84)
  [120/281] pancreas_200 shape=(321, 321, 89)
  [150/281] pancreas_243 shape=(260, 260, 102)
  [180/281] pancreas_286 shape=(262, 262, 93)
  [210/281] pancreas_323 shape=(327, 327, 95)
  [240/281] pancreas_365 shape=(299, 299, 188)
  [270/281] pancreas_406 shape=(305, 305, 86)

✅ 281 cached files ready


In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.residual = nn.Identity() if in_ch == out_ch else nn.Conv3d(in_ch, out_ch, 1, bias=False)
    def forward(self, x): return self.conv(x) + self.residual(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.down = nn.Conv3d(in_ch, out_ch, 2, stride=2, bias=False)
        self.conv = ConvBlock(out_ch, out_ch, dropout)
    def forward(self, x): return self.conv(self.down(x))

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch, dropout)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class PatchEmbedding3D(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size=2):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.proj(x); B,C,D,H,W = x.shape
        x = rearrange(x, 'b c d h w -> b (d h w) c')
        return self.norm(x), (D,H,W)

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=6, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim*mlp_ratio)), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(dim*mlp_ratio), dim), nn.Dropout(dropout),
        )
    def forward(self, x):
        h = self.norm1(x); x = x + self.attn(h,h,h)[0]
        return x + self.mlp(self.norm2(x))

class ViTBottleneck(nn.Module):
    def __init__(self, in_ch, embed_dim=384, heads=6, depth=3, patch_size=2, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding3D(in_ch, embed_dim, patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, 27, embed_dim)*0.02)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, heads, dropout=dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.proj_back = nn.Linear(embed_dim, in_ch)
    def forward(self, x):
        B,C,D,H,W = x.shape
        tokens, (Dp,Hp,Wp) = self.patch_embed(x)
        N = tokens.shape[1]
        pos = F.interpolate(self.pos_embed.transpose(1,2), size=N, mode='linear', align_corners=False).transpose(1,2) if N!=27 else self.pos_embed
        tokens = self.blocks(tokens + pos)
        return rearrange(self.proj_back(self.norm(tokens)), 'b (d h w) c -> b c d h w', d=Dp, h=Hp, w=Wp)

class ViTUNet(nn.Module):
    def __init__(self, in_ch=1, num_classes=2, base=24, vit_dim=384, vit_depth=3, vit_heads=6, deep_sup=True):
        super().__init__()
        self.deep_sup = deep_sup
        self.enc1 = ConvBlock(in_ch, base); self.enc2 = DownBlock(base, base*2)
        self.enc3 = DownBlock(base*2, base*4, 0.1); self.enc4 = DownBlock(base*4, base*8, 0.1)
        self.down_bot = nn.Conv3d(base*8, base*8, 2, stride=2, bias=False)
        self.vit = ViTBottleneck(base*8, vit_dim, vit_heads, vit_depth, 2, 0.1)
        self.dec4 = UpBlock(base*8, base*8, base*8, 0.1); self.dec3 = UpBlock(base*8, base*4, base*4, 0.1)
        self.dec2 = UpBlock(base*4, base*2, base*2); self.dec1 = UpBlock(base*2, base, base)
        self.final = nn.Conv3d(base, num_classes, 1)
        if deep_sup:
            self.ds3 = nn.Conv3d(base*4, num_classes, 1)
            self.ds2 = nn.Conv3d(base*2, num_classes, 1)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x):
        s1=self.enc1(x); s2=self.enc2(s1); s3=self.enc3(s2); s4=self.enc4(s3)
        b = self.vit(self.down_bot(s4))
        d4=self.dec4(b,s4); d3=self.dec3(d4,s3); d2=self.dec2(d3,s2); d1=self.dec1(d2,s1)
        out = self.final(d1)
        if self.deep_sup and self.training: return out, self.ds3(d3), self.ds2(d2)
        return out

# Losses
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1); nc = pred.shape[1]
        toh = F.one_hot(target.long(), nc).permute(0,4,1,2,3).float()
        scores = []
        for c in range(1, nc):
            p, t = pred[:,c].flatten(1), toh[:,c].flatten(1)
            scores.append((2*(p*t).sum(1)+self.smooth)/(p.sum(1)+t.sum(1)+self.smooth))
        return 1 - torch.stack(scores).mean()

class DiceCELoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.dice = SoftDiceLoss()
        w = torch.tensor(class_weights).float().cuda() if class_weights else None
        self.ce = nn.CrossEntropyLoss(weight=w)
    def forward(self, pred, target):
        return 0.5 * self.dice(pred, target) + 0.5 * self.ce(pred, target.long())

class DeepSupLoss(nn.Module):
    def __init__(self, base_loss, weights=[1.0, 0.5, 0.25]):
        super().__init__(); self.base = base_loss; self.weights = weights
    def forward(self, outputs, target):
        if not isinstance(outputs, tuple): return self.base(outputs, target)
        total = 0.0
        for pred, w in zip(outputs, self.weights):
            if pred.shape[2:] != target.shape[1:]:
                t = F.interpolate(target.unsqueeze(1).float(), size=pred.shape[2:], mode='nearest').squeeze(1).long()
            else: t = target
            total += w * self.base(pred, t)
        return total

class PolyLRScheduler:
    def __init__(self, opt, max_ep, power=0.9):
        self.opt=opt; self.max_ep=max_ep; self.power=power
        self.base_lrs=[pg['lr'] for pg in opt.param_groups]
    def step(self, ep):
        f = (1-ep/self.max_ep)**self.power
        for pg, blr in zip(self.opt.param_groups, self.base_lrs): pg['lr']=blr*f

def compute_dice(pred, target, num_classes=2):
    if pred.ndim == 5: pred = pred.argmax(dim=1)
    scores = {}
    for c in range(1, num_classes):
        p = (pred==c).float().flatten(1); t = (target==c).float().flatten(1)
        inter = (p*t).sum(1); union = p.sum(1)+t.sum(1); mask = union > 0
        if mask.sum() > 0: scores[f"c{c}"] = (2*inter[mask]/(union[mask]+1e-8)).mean().item()
        else: scores[f"c{c}"] = float('nan')
    valid = [v for v in scores.values() if v==v]
    scores["mean"] = sum(valid)/len(valid) if valid else 0.0
    return scores

def keep_largest_component(mask):
    if mask.sum() == 0: return mask
    labeled, num = scipy_label(mask)
    if num <= 1: return mask
    sizes = [(labeled==i).sum() for i in range(1, num+1)]
    return (labeled == (np.argmax(sizes)+1)).astype(mask.dtype)

# Augmentation
def elastic_deform(image, label, alpha=80, sigma=8):
    shape = image.shape
    dx = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    dy = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    dz = gaussian_filter(np.random.randn(*shape)*alpha, sigma, mode='constant')
    z,y,x = np.meshgrid(np.arange(shape[0]),np.arange(shape[1]),np.arange(shape[2]),indexing='ij')
    coords = [np.clip(z+dz,0,shape[0]-1),np.clip(y+dy,0,shape[1]-1),np.clip(x+dx,0,shape[2]-1)]
    return map_coordinates(image,coords,order=1,mode='reflect').astype(np.float32), \
           map_coordinates(label.astype(float),coords,order=0,mode='reflect').astype(np.int32)

def gamma_aug(image, gamma_range=(0.7,1.5)):
    mn=image.min(); s=image-mn+1e-8; mx=s.max()+1e-8
    g=np.random.uniform(*gamma_range); out=np.power(s/mx,g)*mx+mn
    if np.random.random()<0.5: out=-out+image.mean()*2
    return out.astype(np.float32)

print(f"✅ Model: {sum(p.numel() for p in ViTUNet(1,2,24).parameters())/1e6:.1f}M params")

✅ Model: 13.7M params


In [6]:
class CachedDataset(Dataset):
    def __init__(self, files, patch_size, augment=True, fg_rate=0.7, binary=True):
        self.files = files; self.ps = patch_size
        self.augment = augment; self.fg_rate = fg_rate; self.binary = binary
        print(f"Dataset: {len(files)} volumes, patch={patch_size}, aug={augment}")

    def __len__(self):
        return len(self.files) * 5

    def __getitem__(self, idx):
        data = np.load(self.files[idx % len(self.files)])
        ct = data['image']; label = data['label'].astype(np.int32)
        if self.binary: label = (label > 0).astype(np.int32)
        img, lbl = self._patch(ct, label)
        if self.augment: img, lbl = self._aug(img, lbl)
        return torch.from_numpy(img).unsqueeze(0).float(), torch.from_numpy(lbl).long()

    def _patch(self, image, label):
        D,H,W = image.shape; pd,ph,pw = self.ps
        if np.random.random() < self.fg_rate:
            fg = np.argwhere(label > 0)
            if len(fg) > 0:
                c = fg[np.random.randint(len(fg))] + np.random.randint(-pd//4, pd//4+1, size=3)
                d = np.clip(c[0]-pd//2, 0, max(0,D-pd))
                h = np.clip(c[1]-ph//2, 0, max(0,H-ph))
                w = np.clip(c[2]-pw//2, 0, max(0,W-pw))
            else:
                d,h,w = [np.random.randint(0,max(1,s-p+1)) for s,p in zip(image.shape,self.ps)]
        else:
            d,h,w = [np.random.randint(0,max(1,s-p+1)) for s,p in zip(image.shape,self.ps)]
        ip = image[d:d+pd, h:h+ph, w:w+pw]
        lp = label[d:d+pd, h:h+ph, w:w+pw]
        if ip.shape != tuple(self.ps):
            pi,pl = np.zeros(self.ps,np.float32), np.zeros(self.ps,np.int32)
            s=ip.shape; pi[:s[0],:s[1],:s[2]]=ip; pl[:s[0],:s[1],:s[2]]=lp
            ip,lp = pi,pl
        return ip, lp

    def _aug(self, img, lbl):
        for ax in range(3):
            if np.random.random()<0.5: img=np.flip(img,ax).copy(); lbl=np.flip(lbl,ax).copy()
        if np.random.random()<0.3:
            ang=np.random.uniform(-15,15); axes=[(0,1),(0,2),(1,2)][np.random.randint(3)]
            img=rotate(img,ang,axes=axes,reshape=False,order=1,mode='reflect')
            lbl=rotate(lbl.astype(float),ang,axes=axes,reshape=False,order=0,mode='reflect').astype(np.int32)
        if np.random.random()<0.2: img,lbl=elastic_deform(img,lbl)
        if np.random.random()<0.3: img=gamma_aug(img)
        if np.random.random()<0.2: img=img+np.random.normal(0,0.02,img.shape).astype(np.float32)
        if np.random.random()<0.3: img=img*np.random.uniform(0.9,1.1)+np.random.uniform(-0.1,0.1)
        if np.random.random()<0.15: img=gaussian_filter(img,sigma=np.random.uniform(0.5,1.0))
        return img.astype(np.float32), lbl.astype(np.int32)

print("✅ Dataset class ready")

✅ Dataset class ready


In [7]:
def train_stage1():
    # Split data
    np.random.seed(42)
    idx = np.random.permutation(len(cached_files))
    vs = int(len(cached_files) * CONFIG["val_split"])
    vf = [cached_files[i] for i in idx[:vs]]
    tf = [cached_files[i] for i in idx[vs:]]

    print(f"📂 Dataset Split: {len(tf)} Train | {len(vf)} Validation")

    tds = CachedDataset(tf, CONFIG["patch_size"], True, CONFIG["fg_sample_rate"], binary=True)
    vds = CachedDataset(vf, CONFIG["patch_size"], False, 0.5, binary=True)
    tl = DataLoader(tds, CONFIG["batch_size"], True, num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
    vl = DataLoader(vds, CONFIG["batch_size"], False, num_workers=CONFIG["num_workers"], pin_memory=True)

    # Model
    model = ViTUNet(1, 2, 24, 384, 3, 6).to(device)
    criterion = DeepSupLoss(DiceCELoss(class_weights=[0.3, 0.7]))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = PolyLRScheduler(optimizer, CONFIG["epochs"])
    scaler = GradScaler()

    start_epoch = 0
    best_dice = 0.0
    hist = {"loss": [], "dice_raw": [], "dice_pp": []}

    # ============================================================
    # HARDCODED RESUME - LOADS CHECKPOINT DIRECTLY
    # ============================================================
    RESUME_FILE = CHECKPOINT_PATH  # From Cell 2 auto-detection

    if RESUME_FILE and os.path.exists(RESUME_FILE):
        print(f"🔄 Loading checkpoint: {RESUME_FILE}")
        ck = torch.load(RESUME_FILE, map_location=device)
        model.load_state_dict(ck['model_state_dict'])
        optimizer.load_state_dict(ck['optimizer_state_dict'])
        scaler.load_state_dict(ck['scaler_state_dict'])
        start_epoch = ck['epoch'] + 1
        best_dice = ck['best_dice']
        hist = ck.get('history', hist)
        print(f"✅ Resumed from Epoch {start_epoch}. Best Dice: {best_dice:.4f}")
    else:
        print("🆕 No checkpoint found. Starting from scratch.")

    print(f"\n{'='*60}")
    print(f"  STAGE 1: PANCREAS LOCALIZATION")
    print(f"  Epochs: {start_epoch} -> {CONFIG['epochs']}")
    print(f"{'='*60}\n")

    # ============================================================
    # TRAINING LOOP
    # ============================================================
    for epoch in range(start_epoch, CONFIG["epochs"]):
        t0 = time.time()
        model.train()
        rl = 0
        optimizer.zero_grad()

        for i, (imgs, lbls) in enumerate(tl):
            imgs, lbls = imgs.to(device), lbls.to(device)
            with autocast():
                loss = criterion(model(imgs), lbls) / CONFIG["accum_steps"]
            scaler.scale(loss).backward()
            if (i+1) % CONFIG["accum_steps"] == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            rl += loss.item() * CONFIG["accum_steps"]

        scheduler.step(epoch)
        tl_loss = rl / max(len(tl), 1)

        # Validation
        do_val = (epoch % 5 == 0) or (epoch >= CONFIG["epochs"] - 50)
        vdr, vdp = 0.0, 0.0

        if do_val:
            model.eval()
            dr_sum, dp_sum, cnt = 0, 0, 0
            with torch.no_grad():
                for imgs, lbls in vl:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    with autocast():
                        pred = model(imgs)
                        if isinstance(pred, tuple): pred = pred[0]
                    d = compute_dice(pred, lbls, 2)
                    dr_sum += d["mean"]
                    pn = pred.argmax(1).cpu().numpy()
                    ppd = 0
                    for b in range(pn.shape[0]):
                        pp = keep_largest_component((pn[b]>0).astype(np.uint8))
                        pt = torch.from_numpy(pp).unsqueeze(0).to(device)
                        ppd += compute_dice(pt, lbls[b:b+1], 2)["mean"]
                    dp_sum += ppd / pn.shape[0]
                    cnt += 1
            vdr = dr_sum / max(cnt, 1)
            vdp = dp_sum / max(cnt, 1)
        else:
            vdr = hist["dice_raw"][-1] if hist["dice_raw"] else 0
            vdp = hist["dice_pp"][-1] if hist["dice_pp"] else 0

        hist["loss"].append(tl_loss)
        hist["dice_raw"].append(vdr)
        hist["dice_pp"].append(vdp)
        el = time.time() - t0
        lr = optimizer.param_groups[0]['lr']

        # Save best
        star = ""
        if do_val and vdp > best_dice:
            best_dice = vdp
            torch.save({
                'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'best_dice': best_dice, 'history': hist
            }, os.path.join(OUTPUT_DIR, "stage1_best.pth"))
            star = " ★"

        # Save latest EVERY epoch
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_dice': best_dice, 'history': hist
        }, os.path.join(OUTPUT_DIR, "stage1_latest.pth"))

        if do_val or star:
            print(f"E{epoch:03d} ({el:.0f}s) Loss:{tl_loss:.4f} Raw:{vdr:.4f} PP:{vdp:.4f} LR:{lr:.2e}{star}")

        # Periodic checkpoint
        if (epoch+1) % 50 == 0:
            torch.save({
                'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'best_dice': best_dice, 'history': hist
            }, os.path.join(OUTPUT_DIR, f"stage1_ep{epoch}.pth"))
            print(f"  💾 Saved stage1_ep{epoch}.pth")

    print(f"\n✅ Training Complete! Best Dice: {best_dice:.4f}")
    return hist

# RUN IT
history = train_stage1()

📂 Dataset Split: 239 Train | 42 Validation
Dataset: 239 volumes, patch=(96, 96, 96), aug=True
Dataset: 42 volumes, patch=(96, 96, 96), aug=False
🔄 Loading checkpoint: /kaggle/input/models/logiverse/model1ep99/pytorch/default/1/stage1_ep99.pth
✅ Resumed from Epoch 100. Best Dice: 0.5112

  STAGE 1: PANCREAS LOCALIZATION
  Epochs: 100 -> 250

E100 (281s) Loss:0.3943 Raw:0.6571 PP:0.5994 LR:6.31e-04 ★
E105 (281s) Loss:0.3135 Raw:0.6648 PP:0.6487 LR:6.12e-04 ★
E110 (282s) Loss:0.2942 Raw:0.6588 PP:0.6199 LR:5.93e-04
E115 (283s) Loss:0.2725 Raw:0.6465 PP:0.6222 LR:5.74e-04
E120 (281s) Loss:0.2560 Raw:0.6821 PP:0.6384 LR:5.55e-04
E125 (283s) Loss:0.2543 Raw:0.7333 PP:0.6743 LR:5.36e-04 ★
E130 (281s) Loss:0.2476 Raw:0.6859 PP:0.6396 LR:5.17e-04
E135 (279s) Loss:0.2486 Raw:0.6947 PP:0.6560 LR:4.97e-04
E140 (282s) Loss:0.2400 Raw:0.6956 PP:0.6643 LR:4.78e-04
E145 (281s) Loss:0.2430 Raw:0.6732 PP:0.6407 LR:4.58e-04
  💾 Saved stage1_ep149.pth
E150 (281s) Loss:0.2323 Raw:0.6850 PP:0.6643 LR:4.38e-

KeyboardInterrupt: 